In [1]:
import pymysql
from neo4j import GraphDatabase, basic_auth
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
from datetime import datetime, date
from decimal import Decimal

# --- 구성 (Configuration) ---
MAX_WORKERS = 5
BATCH_SIZE = 5000

MYSQL_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "dev",
    "password": "pwd",
    "database": "orders",
}

NEO4J_CONFIG = {
    "uri": "bolt://localhost:7687",
    "username": "neo4j",
    "password": "gustjs21@", # 실제 Neo4j 비밀번호로 교체하세요!
    "database": 'test03', # 현재 사용 중인 데이터베이스 이름
}

# --- 로깅 설정 (Logging Setup) ---
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

file_handler = logging.FileHandler('migration.log')
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# --- 헬퍼 함수 (Helper Functions) ---

def clean_property_name(name):
    cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
    if cleaned_name and not re.match(r'^[a-zA-Z_]', cleaned_name[0]):
        cleaned_name = '_' + cleaned_name
    if not cleaned_name or cleaned_name == '_':
        return f"prop_{name}"
    return cleaned_name

def custom_json_serializer(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Decimal):
        return float(obj)
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

def clear_neo4j_database(driver):
    logger.info("Neo4j 데이터베이스를 초기화 중입니다...")
    try:
        with driver.session(database=NEO4J_CONFIG["database"]) as session:
            session.run("CALL apoc.periodic.iterate('MATCH (n) RETURN n', 'DETACH DELETE n', {batchSize: 10000, parallel: true})")
        logger.info("Neo4j 데이터베이스 초기화 완료.")
    except Exception as e:
        logger.error(f"Neo4j 데이터베이스 초기화 중 오류 발생: {e}", exc_info=True)


def create_constraints(driver, node_label, primary_key_column):
    neo4j_pk_column = clean_property_name(primary_key_column)
    try:
        with driver.session(database=NEO4J_CONFIG["database"]) as session:
            cypher_query = f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{node_label}) REQUIRE n.{neo4j_pk_column} IS UNIQUE"
            session.run(cypher_query)
            logger.info(f"'{node_label}' 레이블의 '{neo4j_pk_column}' 속성에 대한 고유성 제약 조건 생성 완료.")
    except Exception as e:
        logger.warning(f"'{node_label}' 레이블의 '{neo4j_pk_column}' 속성에 대한 제약 조건 생성 실패: {e}. (쿼리: {cypher_query})", exc_info=True)

def create_nodes_batch(driver, label, records, primary_key_col_name):
    logger.debug(f"Neo4j에 '{label}' 노드 {len(records)}개 생성/병합 중...")
    try:
        with driver.session(database=NEO4J_CONFIG["database"]) as session:
            cypher = (
                f"UNWIND $rows AS row\n"
                f"MERGE (n:{label} {{ {primary_key_col_name}: row.{primary_key_col_name} }})\n"
                f"SET n = row"
            )
            session.run(cypher, rows=records)
        logger.debug(f"'{label}' 노드 {len(records)}개 생성/병합 완료.")
    except Exception as e:
        logger.error(f"'{label}' 노드 생성/병합 중 오류 발생 (배치 크기: {len(records)}): {e}", exc_info=True)


def migrate_nodes_batched(mysql_conn, driver, table_name, node_label, primary_key_column_name):
    offset = 0
    total_processed = 0
    cursor = mysql_conn.cursor(pymysql.cursors.DictCursor)
    futures = []
    
    logger.info(f"테이블 '{table_name}'에서 '{node_label}' 노드로 마이그레이션 시작...")

    cursor.execute(f"SHOW COLUMNS FROM `{table_name}` LIKE 'deletedAt'")
    has_deleted_at = cursor.fetchone() is not None

    where_clause = " WHERE deletedAt IS NULL" if has_deleted_at else ""

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        while True:
            cursor.execute(f"SELECT * FROM `{table_name}`{where_clause} LIMIT {BATCH_SIZE} OFFSET {offset}")
            rows = cursor.fetchall()
            if not rows:
                break

            processed_rows = []
            for row in rows:
                cleaned_row = {}
                for k, v in row.items():
                    cleaned_k = clean_property_name(k)
                    if isinstance(v, Decimal):
                        cleaned_row[cleaned_k] = float(v)
                    elif isinstance(v, (datetime, date)):
                        cleaned_row[cleaned_k] = v.isoformat()
                    elif isinstance(v, (dict, list)):
                        cleaned_row[cleaned_k] = json.dumps(v, default=custom_json_serializer)
                    else:
                        cleaned_row[cleaned_k] = v
                processed_rows.append(cleaned_row)

            futures.append(executor.submit(create_nodes_batch, driver, node_label, processed_rows, clean_property_name(primary_key_column_name)))
            offset += BATCH_SIZE
            logger.info(f"테이블 '{table_name}'에서 {len(rows)}개의 레코드 배치 제출. 총 제출된 레코드: {offset}")

        for future in as_completed(futures):
            try:
                future.result() 
                total_processed += BATCH_SIZE 
            except Exception as exc:
                logger.error(f"'{node_label}' 노드 생성 중 치명적인 오류 발생: {exc}", exc_info=True)
    logger.info(f"테이블 '{table_name}'의 '{node_label}' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): {total_processed}")


def migrate_relationship(mysql_conn, driver, from_table, from_label, mysql_from_key, to_label, mysql_to_key, relationship_type):
    offset = 0
    total_relationships_created = 0
    cursor = mysql_conn.cursor(pymysql.cursors.DictCursor)

    neo4j_from_key = clean_property_name(mysql_from_key)
    neo4j_to_key = clean_property_name(mysql_to_key)

    logger.info(f"관계 마이그레이션 시작: {from_label}.{mysql_from_key} -[{relationship_type}]-> {to_label}.{mysql_to_key}")

    cursor.execute(f"SHOW COLUMNS FROM `{from_table}` LIKE 'deletedAt'")
    has_deleted_at = cursor.fetchone() is not None

    where_clause = f" WHERE `{mysql_from_key}` IS NOT NULL AND `{mysql_to_key}` IS NOT NULL"
    if has_deleted_at:
        where_clause += " AND deletedAt IS NULL"

    with driver.session(database=NEO4J_CONFIG["database"]) as session:
        while True:
            # MySQL에서 관계 생성을 위한 ID 쌍 가져오기
            query = f"SELECT `{mysql_from_key}`, `{mysql_to_key}` FROM `{from_table}`{where_clause} LIMIT {BATCH_SIZE} OFFSET {offset}"
            cursor.execute(query)
            rows = cursor.fetchall()

            if not rows:
                break

            cypher_query = f"""
            UNWIND $batch AS rel
            MATCH (a:{from_label} {{ {neo4j_from_key}: rel.`{mysql_from_key}` }})
            MATCH (b:{to_label} {{ {neo4j_to_key}: rel.`{mysql_to_key}` }})
            MERGE (a)-[:{relationship_type}]->(b)
            """
            try:
                session.run(cypher_query, batch=rows)
                total_relationships_created += len(rows)
                logger.info(f"관계 배치 처리: {len(rows)}개 ({from_label}-[:{relationship_type}]->{to_label}). 총 {total_relationships_created}개.")
            except Exception as e:
                logger.error(f"관계 생성 중 오류 발생 ({from_label}-[:{relationship_type}]->{to_label}, 배치 크기: {len(rows)}): {e}", exc_info=True)

            offset += BATCH_SIZE
    logger.info(f"관계 마이그레이션 완료: {from_label}-[:{relationship_type}]->{to_label}. 총 {total_relationships_created}개의 관계 생성.")


def main():
    logger.info("🔥 마이그레이션 프로세스 시작 🔥")
    mysql_conn = None
    neo4j_driver = None
    try:
        logger.info("MySQL 데이터베이스에 연결 중...")
        mysql_conn = pymysql.connect(**MYSQL_CONFIG, charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor)
        logger.info("MySQL 연결 성공.")

        logger.info("Neo4j 데이터베이스에 연결 중...")
        neo4j_driver = GraphDatabase.driver(
            NEO4J_CONFIG["uri"], auth=basic_auth(NEO4J_CONFIG["username"], NEO4J_CONFIG["password"])
        )
        neo4j_driver.verify_connectivity()
        logger.info("Neo4j 연결 성공.")

        clear_neo4j_database(neo4j_driver)

        table_node_config = {
            "categories": ["Category", "id"],
            "countries": ["Country", "alpha2code"], # Country의 PK는 그대로 'alpha2code'로 유지
            "coupon_usages": ["CouponUsage", "id"],
            "coupons": ["Coupon", "id"],
            "customers": ["Customer", "id"],
            "food_courts": ["FoodCourt", "id"],
            "grouped_texts": ["GroupedText", "id"],
            "inventory_items": ["InventoryItem", "id"],
            "inventory_transactions": ["InventoryTransaction", "id"],
            "locales": ["Locale", "code"],
            "menu_option_choice_to_menu_options": ["MenuOptionChoiceToMenuOption", "id"],
            "menu_option_choices": ["MenuOptionChoice", "id"],
            "menu_option_to_menus": ["MenuOptionToMenu", "id"],
            "menu_options": ["MenuOption", "id"],
            "menu_to_inventory_items": ["MenuToInventoryItem", "id"],
            "menus": ["Menu", "id"],
            "order_items": ["OrderItem", "id"],
            "orders": ["Order", "id"],
            "payment_option_enums": ["PaymentOptionEnum", "paymentOptionEnum"],
            "payments": ["Payment", "id"],
            "refund_items": ["RefundItem", "id"],
            "refunds": ["Refund", "id"],
            "store_attributes": ["StoreAttribute", "id"],
            "store_meta_advert_insights": ["StoreMetaAdvertInsight", "id"],
            "store_meta_adverts": ["StoreMetaAdvert", "id"],
            "store_tables": ["StoreTable", "id"],
            "stores": ["Store", "id"],
        }

        logger.info("--- 노드 마이그레이션 및 제약 조건 생성 시작 ---")
        for mysql_table, (neo4j_label, mysql_pk_column_name) in table_node_config.items():
            create_constraints(neo4j_driver, neo4j_label, mysql_pk_column_name)
            migrate_nodes_batched(mysql_conn, neo4j_driver, mysql_table, neo4j_label, mysql_pk_column_name)
        logger.info("--- 노드 마이그레이션 및 제약 조건 생성 완료 ---")

        logger.info("--- 관계 마이그레이션 시작 ---")
        relationships_to_migrate = [
            # categories 테이블 (Category 노드)
            ("categories", "Category", "groupedTextId", "GroupedText", "id", "HAS_TEXT_DESCRIPTION"),
            ("categories", "Category", "storeId", "Store", "id", "BELONONGS_TO_STORE"),

            # coupon_usages 테이블 (CouponUsage 노드)
            ("coupon_usages", "CouponUsage", "couponId", "Coupon", "id", "USED_COUPON"),
            ("coupon_usages", "CouponUsage", "customerId", "Customer", "id", "USED_BY"),
            ("coupon_usages", "CouponUsage", "orderId", "Order", "id", "APPLIED_TO_ORDER"),

            # coupons 테이블 (Coupon 노드)
            ("coupons", "Coupon", "storeId", "Store", "id", "ISSUED_BY"),

            # food_courts 테이블과 countries 테이블 간의 관계는 제외됨.
            # ("food_courts", "FoodCourt", "countryCode", "Country", "alpha2code", "LOCATED_IN_COUNTRY"), # <<< 이 라인 제거

            # grouped_texts 테이블 (GroupedText 노드)
            ("grouped_texts", "GroupedText", "originalLocaleCode", "Locale", "code", "HAS_ORIGINAL_LOCALE"),

            # inventory_items 테이블 (InventoryItem 노드)
            ("inventory_items", "InventoryItem", "storeId", "Store", "id", "STOCKED_IN"),

            # inventory_transactions 테이블 (InventoryTransaction 노드)
            ("inventory_transactions", "InventoryTransaction", "inventoryItemId", "InventoryItem", "id", "AFFECTS_ITEM"),
            ("inventory_transactions", "InventoryTransaction", "storeId", "Store", "id", "OCCURRED_AT_STORE"),

            # menu_option_choice_to_menu_options 테이블 (중간 테이블: MenuOptionChoiceToMenuOption 노드)
            ("menu_option_choice_to_menu_options", "MenuOptionChoiceToMenuOption", "menuOptionChoiceId", "MenuOptionChoice", "id", "HAS_CHOICE_REF"),
            ("menu_option_choice_to_menu_options", "MenuOptionChoiceToMenuOption", "menuOptionId", "MenuOption", "id", "HAS_OPTION_REF"),
            
            # menu_option_choices 테이블 (MenuOptionChoice 노드)
            ("menu_option_choices", "MenuOptionChoice", "storeId", "Store", "id", "BELONGS_TO_STORE_MOC"),
            ("menu_option_choices", "MenuOptionChoice", "groupedTextId", "GroupedText", "id", "HAS_NAME_TEXT_MOC"),

            # menu_option_to_menus 테이블 (중간 테이블: MenuOptionToMenu 노드)
            ("menu_option_to_menus", "MenuOptionToMenu", "menuId", "Menu", "id", "ASSOCIATED_WITH_MENU"),
            ("menu_option_to_menus", "MenuOptionToMenu", "menuOptionId", "MenuOption", "id", "ASSOCIATED_WITH_OPTION"),
            
            # menu_options 테이블 (MenuOption 노드)
            ("menu_options", "MenuOption", "storeId", "Store", "id", "BELONGS_TO_STORE_MO"),
            ("menu_options", "MenuOption", "groupedTextId", "GroupedText", "id", "HAS_NAME_TEXT_MO"),

            # menu_to_inventory_items 테이블 (중간 테이블: MenuToInventoryItem 노드)
            ("menu_to_inventory_items", "MenuToInventoryItem", "menuId", "Menu", "id", "LINKED_TO_MENU"),
            ("menu_to_inventory_items", "MenuToInventoryItem", "inventoryItemId", "InventoryItem", "id", "LINKED_TO_INVENTORY_ITEM"),
            
            # menus 테이블 (Menu 노드)
            ("menus", "Menu", "storeId", "Store", "id", "OFFERED_BY"),
            ("menus", "Menu", "categoryId", "Category", "id", "BELONGS_TO_CATEGORY"),
            ("menus", "Menu", "groupedTextId", "GroupedText", "id", "HAS_NAME_TEXT_MENU"),

            # order_items 테이블 (OrderItem 노드)
            ("order_items", "OrderItem", "orderId", "Order", "id", "PART_OF"),
            ("order_items", "OrderItem", "menuId", "Menu", "id", "REFERENCES"),

            # orders 테이블 (Order 노드)
            ("orders", "Order", "customerId", "Customer", "id", "PLACED_BY"),
            ("orders", "Order", "storeId", "Store", "id", "PLACED_AT"),
            ("orders", "Order", "paymentOptionEnum", "PaymentOptionEnum", "paymentOptionEnum", "USES_PAYMENT_OPTION"),
            ("orders", "Order", "storeTableId", "StoreTable", "id", "RESERVED_TABLE"),
            ("orders", "Order", "paymentId", "Payment", "id", "HAS_PAYMENT"),

            # payments 테이블 (Payment 노드)
            ("payments", "Payment", "orderId", "Order", "id", "FOR_ORDER"),
            ("payments", "Payment", "customerId", "Customer", "id", "MADE_BY_CUSTOMER"),
            ("payments", "Payment", "storeId", "Store", "id", "MADE_AT_STORE"),

            # refund_items 테이블 (RefundItem 노드)
            ("refund_items", "RefundItem", "refundId", "Refund", "id", "IS_PART_OF_REFUND"),
            ("refund_items", "RefundItem", "orderItemId", "OrderItem", "id", "REFUNDS_ITEM"),

            # refunds 테이블 (Refund 노드)
            ("refunds", "Refund", "orderId", "Order", "id", "FOR_ORDER_REFUND"),
            ("refunds", "Refund", "paymentId", "Payment", "id", "RELATED_TO_PAYMENT_REFUND"),
            ("refunds", "Refund", "storeId", "Store", "id", "INITIATED_BY_STORE"),

            # store_attributes 테이블 (StoreAttribute 노드)
            ("store_attributes", "StoreAttribute", "storeId", "Store", "id", "FOR_STORE"),

            # store_meta_advert_insights 테이블 (StoreMetaAdvertInsight 노드)
            ("store_meta_advert_insights", "StoreMetaAdvertInsight", "storeMetaAdvertId", "StoreMetaAdvert", "id", "HAS_INSIGHT"),

            # store_meta_adverts 테이블 (StoreMetaAdvert 노드)
            ("store_meta_adverts", "StoreMetaAdvert", "storeId", "Store", "id", "FOR_STORE_ADVERTISEMENT"),

            # stores 테이블 (Store 노드)
            ("stores", "Store", "foodCourtId", "FoodCourt", "id", "LOCATED_IN_FOODCOURT"), # 이 관계는 그대로 유지
        ]

        for rel_info in relationships_to_migrate:
            from_table, from_label, mysql_from_key, to_label, mysql_to_key, relationship_type = rel_info
            migrate_relationship(mysql_conn, neo4j_driver, from_table, from_label, mysql_from_key, to_label, mysql_to_key, relationship_type)

        logger.info("--- 관계 마이그레이션 완료 ---")
        logger.info("✅ 마이그레이션 프로세스 전체 완료 ✅")

    except pymysql.Error as e:
        logger.error(f"MySQL 연결 또는 쿼리 오류: {e}", exc_info=True)
    except Exception as e:
        logger.error(f"예상치 못한 오류 발생: {e}", exc_info=True)
    finally:
        if mysql_conn:
            mysql_conn.close()
            logger.info("MySQL 연결 종료.")
        if neo4j_driver:
            neo4j_driver.close()
            logger.info("Neo4j 연결 종료.")

if __name__ == "__main__":
    main()

2025-07-09 17:51:28,140 - INFO - 🔥 마이그레이션 프로세스 시작 🔥
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\logging\__init__.py", line 1086, in emit
    stream.write(msg + self.terminator)
UnicodeEncodeError: 'cp949' codec can't encode character '\U0001f525' in position 44: illegal multibyte sequence
Call stack:
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\ipykernel\kernelapp.py", line 

2025-07-09 17:51:59,371 - INFO - 'InventoryTransaction' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:51:59,373 - INFO - 테이블 'inventory_transactions'에서 'InventoryTransaction' 노드로 마이그레이션 시작...
2025-07-09 17:51:59,397 - INFO - 테이블 'inventory_transactions'에서 186개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:51:59,413 - INFO - 테이블 'inventory_transactions'의 'InventoryTransaction' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:51:59,416 - INFO - 'Locale' 레이블의 'code' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:51:59,418 - INFO - 테이블 'locales'에서 'Locale' 노드로 마이그레이션 시작...
2025-07-09 17:51:59,441 - INFO - 테이블 'locales'에서 183개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:51:59,464 - INFO - 테이블 'locales'의 'Locale' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:51:59,468 - INFO - 'MenuOptionChoiceToMenuOption' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:51:59,470 - INFO - 테이블 'menu_option_choice_to_menu_options'에서 'MenuOptionChoiceToMenuOption' 노드로 마이그레이션 시작...
2025-07-09 17:51:59,557 - INFO - 테이블 

2025-07-09 17:53:37,782 - INFO - 테이블 'refund_items'에서 1137개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:53:37,891 - INFO - 테이블 'refund_items'의 'RefundItem' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:53:37,897 - INFO - 'Refund' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:53:37,899 - INFO - 테이블 'refunds'에서 'Refund' 노드로 마이그레이션 시작...
2025-07-09 17:53:37,987 - INFO - 테이블 'refunds'에서 943개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:53:38,166 - INFO - 테이블 'refunds'의 'Refund' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:53:38,171 - INFO - 'StoreAttribute' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17:53:38,173 - INFO - 테이블 'store_attributes'에서 'StoreAttribute' 노드로 마이그레이션 시작...
2025-07-09 17:53:38,285 - INFO - 테이블 'store_attributes'에서 1453개의 레코드 배치 제출. 총 제출된 레코드: 5000
2025-07-09 17:53:38,429 - INFO - 테이블 'store_attributes'의 'StoreAttribute' 노드 마이그레이션 완료. 총 처리된 노드 수 (대략): 5000
2025-07-09 17:53:38,433 - INFO - 'StoreMetaAdvertInsight' 레이블의 'id' 속성에 대한 고유성 제약 조건 생성 완료.
2025-07-09 17: